# Whisper Transcription (Colab)

Transcribes audio files using **faster-whisper** (`large-v3`) and maps speakers from diarization output.

**Strategy**: Transcribes the **full audio** (not per-chunk) for better quality — Whisper keeps full context across the file. Speakers are assigned by overlapping Whisper segments with diarization turns from `grouped.json`.

**Folder structure** (shared with YouTube Downloader + Diarization notebooks)
```
/jianglens/
  _hf_home/                          ← cached Whisper + pyannote models
  youtube/
    <channel_id>/
      <video_id>/
        audio.wav                    ← input
        dump.json                    ← diarization (raw)
        grouped.json                 ← diarization (grouped)
        transcription.json           ← output (this notebook)
```

**Flow**
1. Run **Config** (mount Drive, settings)
2. Run **Install** (faster-whisper)
3. Run **Scan** (find videos with diarization but no transcription)
4. Run **Batch** (transcribe all pending)


In [ ]:
#@title 0) Config
#@markdown Mount Drive, set model + paths.

import os, sys
from pathlib import Path

# --------------------
# Data locations
# --------------------
LENS_COMPRESSION_DRIVE_ROOT = "/content/drive/MyDrive/jianglens"
LENS_COMPRESSION_YOUTUBE_ROOT = LENS_COMPRESSION_DRIVE_ROOT + "/youtube"
LENS_COMPRESSION_WORK_ROOT = "/content/jianglens"  # fast local workspace

# --------------------
# HuggingFace model cache (Drive-backed, shared with diarization)
# --------------------
HF_HOME = LENS_COMPRESSION_DRIVE_ROOT + "/_hf_home"

# --------------------
# Whisper settings
# --------------------
WHISPER_MODEL = "large-v3"       # best quality; alternatives: "large-v3-turbo", "large-v2", "medium"
WHISPER_COMPUTE_TYPE = "float16"  # float16 for GPU, int8 for less VRAM
WHISPER_LANGUAGE = ""             # empty = auto-detect; set "es", "en", etc. to force

# --------------------
# Batch settings
# --------------------
MAX_WORKERS = 2    # parallel transcription workers (each loads ~3GB VRAM; T4=15GB, L4=24GB)
RUN_BATCH = False  # set True to start processing when you run the batch cell

# Optional: test on a single video first
TEST_VIDEO = ""    # e.g. "UCxxxxxxxx/s_8BKJDiwKw"

# --------------------
# Mount Drive
# --------------------
def _is_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

if _is_colab():
    from google.colab import drive
    drive.mount("/content/drive")

Path(LENS_COMPRESSION_YOUTUBE_ROOT).mkdir(parents=True, exist_ok=True)
Path(LENS_COMPRESSION_WORK_ROOT).mkdir(parents=True, exist_ok=True)
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME

PYTHON_BIN = sys.executable

print("\u2705 Config loaded")
print(f"   Model:    {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")
print(f"   Language: {WHISPER_LANGUAGE or 'auto-detect'}")
print(f"   Workers:  {MAX_WORKERS}")
print(f"   HF_HOME:  {HF_HOME}")
print(f"   YouTube:  {LENS_COMPRESSION_YOUTUBE_ROOT}")


In [ ]:
#@title 1) Install faster-whisper

import subprocess

subprocess.run(["pip", "install", "-q", "faster-whisper"], check=True)

print("\u2705 faster-whisper installed")

# Pre-download the model to Drive cache so workers don't all download simultaneously
print(f"Pre-loading {WHISPER_MODEL} into cache ({HF_HOME})...")
from faster_whisper import WhisperModel
_test_model = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
del _test_model
print(f"\u2705 Model cached. Workers will load from Drive cache.")


In [ ]:
#@title 2) Write worker script

from pathlib import Path

worker_script = r"""
import os, sys, json, shutil
from pathlib import Path

LENS_COMPRESSION_YOUTUBE_ROOT = os.environ.get("LENS_COMPRESSION_YOUTUBE_ROOT", "/content/drive/MyDrive/jianglens/youtube")
LENS_COMPRESSION_WORK_ROOT = os.environ.get("LENS_COMPRESSION_WORK_ROOT", "/content/jianglens")
WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "large-v3")
WHISPER_COMPUTE_TYPE = os.environ.get("WHISPER_COMPUTE_TYPE", "float16")
WHISPER_LANGUAGE = os.environ.get("WHISPER_LANGUAGE", "")


def find_speaker(seg_start, seg_end, diar_segments):
    # Find the speaker with maximum time overlap for a given segment.
    best_speaker = None
    best_overlap = 0.0
    for d in diar_segments:
        overlap = max(0, min(seg_end, d["end"]) - max(seg_start, d["start"]))
        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = d["speaker"]
    return best_speaker or "UNKNOWN"


def main(video_path):
    import time as _time
    t_start = _time.time()

    print(f"\U0001f3a4 [Worker] Transcribing: {video_path}")

    drive_dir = Path(LENS_COMPRESSION_YOUTUBE_ROOT) / video_path
    work_dir = Path(LENS_COMPRESSION_WORK_ROOT) / video_path
    work_dir.mkdir(parents=True, exist_ok=True)

    # ── Find audio ─────────────────────────────────────────
    src = drive_dir / "audio.wav"
    if not src.exists():
        src = drive_dir / "video.wav"
    if not src.exists():
        print(f"\u274c Audio not found in: {drive_dir}")
        return 2

    local_wav = work_dir / "audio.wav"
    if not local_wav.exists():
        shutil.copy(src, local_wav)

    # ── Load diarization (optional) ────────────────────────
    diar_segments = []
    grouped_path = drive_dir / "grouped.json"
    if grouped_path.exists():
        grouped = json.loads(grouped_path.read_text(encoding="utf-8"))
        for entry in grouped.get("diarization", []):
            diar_segments.append({
                "speaker": entry.get("speaker", "UNKNOWN"),
                "start": entry.get("start", entry.get("start_time", 0)),
                "end": entry.get("end", entry.get("end_time", 0)),
            })
        print(f"[Worker] Loaded {len(diar_segments)} diarization segments")
    else:
        print("[Worker] No grouped.json found \u2014 transcribing without speaker assignment")

    # ── Load Whisper model ─────────────────────────────────
    import torch
    from faster_whisper import WhisperModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[Worker] Loading {WHISPER_MODEL} on {device} ({WHISPER_COMPUTE_TYPE})...")
    model = WhisperModel(WHISPER_MODEL, device=device, compute_type=WHISPER_COMPUTE_TYPE)

    # ── Transcribe full audio ──────────────────────────────
    print("[Worker] Transcribing...")
    transcribe_opts = {
        "word_timestamps": True,
        "beam_size": 5,
        "best_of": 5,
        "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        "condition_on_previous_text": True,
        "vad_filter": True,
        "vad_parameters": {"min_silence_duration_ms": 500},
    }
    if WHISPER_LANGUAGE:
        transcribe_opts["language"] = WHISPER_LANGUAGE

    segments_iter, info = model.transcribe(str(local_wav), **transcribe_opts)

    detected_lang = info.language
    lang_prob = info.language_probability
    print(f"[Worker] Language: {detected_lang} ({lang_prob:.1%})")

    # ── Collect segments with word timestamps ──────────────
    raw_segments = []
    for seg in segments_iter:
        words = []
        if seg.words:
            for w in seg.words:
                words.append({
                    "word": w.word.strip(),
                    "start": round(w.start, 3),
                    "end": round(w.end, 3),
                    "probability": round(w.probability, 3),
                })

        speaker = "UNKNOWN"
        if diar_segments:
            speaker = find_speaker(seg.start, seg.end, diar_segments)

        raw_segments.append({
            "start": round(seg.start, 3),
            "end": round(seg.end, 3),
            "text": seg.text.strip(),
            "speaker": speaker,
            "words": words,
        })

    # ── Group into speaker turns ───────────────────────────
    # Merge consecutive segments with the same speaker into turns
    turns = []
    for seg in raw_segments:
        if turns and turns[-1]["speaker"] == seg["speaker"]:
            # Extend current turn
            turns[-1]["end"] = seg["end"]
            turns[-1]["text"] += " " + seg["text"]
            turns[-1]["words"].extend(seg["words"])
            turns[-1]["segments"].append({
                "start": seg["start"],
                "end": seg["end"],
                "text": seg["text"],
            })
        else:
            # New turn
            turns.append({
                "speaker": seg["speaker"],
                "start": seg["start"],
                "end": seg["end"],
                "text": seg["text"],
                "words": list(seg["words"]),
                "segments": [{
                    "start": seg["start"],
                    "end": seg["end"],
                    "text": seg["text"],
                }],
            })

    # ── Build output ───────────────────────────────────────
    full_text = " ".join(t["text"] for t in turns)
    output = {
        "language": detected_lang,
        "language_probability": round(lang_prob, 3),
        "duration": round(info.duration, 3),
        "turns": turns,
        "text": full_text,
    }

    # ── Save ───────────────────────────────────────────────
    out_path = work_dir / "transcription.json"
    out_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding="utf-8")

    # Copy to Drive
    shutil.copy(out_path, drive_dir / "transcription.json")

    elapsed = _time.time() - t_start
    n_words = sum(len(t["words"]) for t in turns)
    audio_dur = info.duration
    speed_ratio = audio_dur / elapsed if elapsed > 0 else 0
    print(f"\u2705 [Worker] {video_path}: {len(turns)} turns, {n_words} words, {elapsed:.0f}s ({speed_ratio:.1f}x realtime)")
    return 0


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: transcription_worker.py <channel_id/video_id>")
        sys.exit(1)
    sys.exit(main(sys.argv[1]))
"""

Path("/content/transcription_worker.py").write_text(worker_script, encoding="utf-8")
print("\u2705 Wrote /content/transcription_worker.py")


In [ ]:
#@title 3) Scan pending videos (has grouped.json but no transcription.json)
#@markdown Finds videos that have been diarized but not yet transcribed.

import json
from pathlib import Path
from collections import Counter

youtube_root = Path(LENS_COMPRESSION_YOUTUBE_ROOT)

pending_videos = []  # list of "channel_id/video_id"
already_done = []
no_diarization = []

if youtube_root.exists():
    for channel_dir in sorted(youtube_root.iterdir()):
        if not channel_dir.is_dir() or channel_dir.name.startswith("_"):
            continue
        channel_id = channel_dir.name
        for video_dir in sorted(channel_dir.iterdir()):
            if not video_dir.is_dir() or video_dir.name.startswith("_"):
                continue
            has_audio = (video_dir / "audio.wav").exists() or (video_dir / "video.wav").exists()
            if not has_audio:
                continue
            has_grouped = (video_dir / "grouped.json").exists()
            has_transcription = (video_dir / "transcription.json").exists()
            path = f"{channel_id}/{video_dir.name}"

            if has_transcription:
                already_done.append(path)
            elif has_grouped:
                pending_videos.append(path)
            else:
                no_diarization.append(path)

print(f"\U0001f4c2 YouTube root: {youtube_root}")

# Per-channel breakdown
channel_counts = Counter(v.split("/")[0] for v in pending_videos)
for ch, count in channel_counts.items():
    meta_path = youtube_root / ch / "_channel.json"
    label = ch
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        label = f"{meta.get('input', ch)} ({ch})"
    print(f"   {label}: {count} pending")

print(f"\n\U0001f4dd Pending transcription: {len(pending_videos)}")
print(f"\u2705 Already transcribed: {len(already_done)}")
print(f"\u23f3 Awaiting diarization: {len(no_diarization)}")

if pending_videos:
    print(f"\nFirst 10 pending:")
    for v in pending_videos[:10]:
        print(f"   {v}")


In [ ]:
#@title 4) Batch processing
#@markdown Runs transcription workers in parallel. Each worker loads the Whisper model independently.

import concurrent.futures
import os
import subprocess
import time as _time


def _worker_env():
    env = os.environ.copy()
    env["HF_HOME"] = HF_HOME
    env["LENS_COMPRESSION_YOUTUBE_ROOT"] = LENS_COMPRESSION_YOUTUBE_ROOT
    env["LENS_COMPRESSION_WORK_ROOT"] = LENS_COMPRESSION_WORK_ROOT
    env["WHISPER_MODEL"] = WHISPER_MODEL
    env["WHISPER_COMPUTE_TYPE"] = WHISPER_COMPUTE_TYPE
    env["WHISPER_LANGUAGE"] = WHISPER_LANGUAGE
    return env


def _run_one(video_path):
    t0 = _time.time()
    print(f"\u23f3 [Started] {video_path}")
    cmd = [PYTHON_BIN, "/content/transcription_worker.py", video_path]
    result = subprocess.run(cmd, env=_worker_env(), capture_output=True, text=True)
    elapsed = _time.time() - t0

    if result.returncode == 0:
        # Print the last line of output (summary)
        lines = result.stdout.strip().split("\n")
        summary = lines[-1] if lines else ""
        print(f"\u2705 [{elapsed:.0f}s] {video_path} {summary}")
        return elapsed

    print(f"\u274c [{elapsed:.0f}s] {video_path}")
    out = (result.stdout + result.stderr).strip()
    if out:
        for line in out.split("\n")[-10:]:
            print(f"   {line}")
    return elapsed


def process_videos(videos, *, max_workers):
    print(f"\U0001f680 Batch transcription: {len(videos)} videos, {max_workers} workers")
    print(f"   Model: {WHISPER_MODEL} ({WHISPER_COMPUTE_TYPE})")
    print(f"   VRAM per worker: ~3GB (large-v3 float16)\n")

    batch_start = _time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        times = list(executor.map(_run_one, videos))

    batch_elapsed = _time.time() - batch_start
    n = len(videos)
    avg = sum(times) / n if n else 0

    # Format elapsed time
    hrs, rem = divmod(int(batch_elapsed), 3600)
    mins, secs = divmod(rem, 60)
    elapsed_str = f"{hrs}h{mins:02d}m{secs:02d}s" if hrs else f"{mins}m{secs:02d}s"

    print(f"\n\U0001f3c1 Batch complete!")
    print(f"   Total wall time: {elapsed_str}")
    print(f"   Videos processed: {n}")
    print(f"   Avg per video: {avg:.0f}s")
    print(f"   Throughput: {n / (batch_elapsed / 3600):.0f} videos/hr" if batch_elapsed > 0 else "")


if RUN_BATCH:
    process_videos(pending_videos, max_workers=MAX_WORKERS)
else:
    print("\u23f8\ufe0f Batch disabled. Set RUN_BATCH = True in Config to start.")
    print(f"   {len(pending_videos)} videos waiting")


In [ ]:
#@title 5) Verify on a single video

import json, os, subprocess
from pathlib import Path

assert TEST_VIDEO, "Set TEST_VIDEO in Config (e.g. 'UCxxxxxxxx/s_8BKJDiwKw')"
print(f"\U0001f9ea Testing transcription on: {TEST_VIDEO}")

env = os.environ.copy()
env["HF_HOME"] = HF_HOME
env["LENS_COMPRESSION_YOUTUBE_ROOT"] = LENS_COMPRESSION_YOUTUBE_ROOT
env["LENS_COMPRESSION_WORK_ROOT"] = LENS_COMPRESSION_WORK_ROOT
env["WHISPER_MODEL"] = WHISPER_MODEL
env["WHISPER_COMPUTE_TYPE"] = WHISPER_COMPUTE_TYPE
env["WHISPER_LANGUAGE"] = WHISPER_LANGUAGE

cmd = [PYTHON_BIN, "/content/transcription_worker.py", TEST_VIDEO]
proc = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"Worker failed with exit code {proc.returncode}")

# Show results
out_path = Path(LENS_COMPRESSION_WORK_ROOT) / TEST_VIDEO / "transcription.json"
if not out_path.exists():
    raise FileNotFoundError(f"transcription.json not found: {out_path}")

data = json.loads(out_path.read_text(encoding="utf-8"))
print(f"\n\U0001f4ca Results:")
print(f"   Language: {data.get('language')} ({data.get('language_probability', 0):.1%})")
print(f"   Duration: {data.get('duration', 0):.1f}s")
print(f"   Turns: {len(data.get('turns', []))}")
print(f"   Words: {sum(len(t.get('words', [])) for t in data.get('turns', []))}")

# Show first 3 turns
print(f"\nFirst 3 turns:")
for i, turn in enumerate(data.get("turns", [])[:3]):
    text_preview = turn["text"][:80] + ("..." if len(turn["text"]) > 80 else "")
    print(f"   [{turn['speaker']}] {turn['start']:.1f}s-{turn['end']:.1f}s: {text_preview}")
